# TuttiPaletti — Agent-Analyse

Analyse-Notebook für das **TuttiPaletti**-Projekt
(`C:\Users\rudi\source\repos\TuttiPaletti`).

Wir lassen einen Taskforce-Agent eine reale Aufgabe lösen — eine Mail aus
`test-mails/` gemäß Workflow 1 (Rechnungsrückfrage) bearbeiten — und tracken
dabei drei Dinge:

1. **Context** — wieviele Messages, wieviele Zeichen vor jedem LLM-Call?
2. **Tool-Ergebnisse** — welche Tools werden aufgerufen, was kommt zurück,
   wie wird das Ergebnis weiterverwendet?
3. **Planung** — plant der Agent (Planner-Tool), wie ändert sich der Plan,
   wie verhält er sich abhängig von der Planning-Strategie?

**Setup**:
- Minimales Tool-Set (`file_read`, `file_write`, `glob`, `grep`, `python`)
- System-Prompt = `CLAUDE.md` + Workflow-Skill direkt eingebettet
  (Alternative: nur Bootstrap-Prompt + Agent liest CLAUDE.md selbst — siehe
  Kommentar in der Agent-Build-Cell)
- Workspace-Sandbox auf `TUTTI_ROOT`
- LLM-Credentials wie in pytaskforce gewohnt (Azure / OpenAI in `.env`)


## 1. Setup


In [10]:
import os
import sys
from pathlib import Path

# pytaskforce-Repo-Root finden, damit src/ im Path landet
PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == "notebooks":
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)

src_path = str(PYTF_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# .env laden
try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / ".env")
except ImportError:
    pass

# TuttiPaletti-Projekt
TUTTI_ROOT = Path(r"C:\Users\rudi\source\repos\TuttiPaletti")
assert TUTTI_ROOT.is_dir(), f"TuttiPaletti nicht gefunden: {TUTTI_ROOT}"

# Ausgabeverzeichnisse vorbereiten
(TUTTI_ROOT / "fall-log").mkdir(exist_ok=True)
(TUTTI_ROOT / "drafts").mkdir(exist_ok=True)

print(f"pytaskforce: {PYTF_ROOT}")
print(f"TuttiPaletti: {TUTTI_ROOT}")
print(f"  test-mails: {sorted(p.name for p in (TUTTI_ROOT / 'test-mails').iterdir())}")
print(f"  skills:     {sorted(p.name for p in (TUTTI_ROOT / 'skills').iterdir())}")
print(f"  kontext:    {sorted(p.name for p in (TUTTI_ROOT / 'kontext').iterdir())}")


pytaskforce: C:\Users\rudi\source\pytaskforce
TuttiPaletti: C:\Users\rudi\source\repos\TuttiPaletti
  test-mails: ['01-mueller-bau-rechnungsfrage.eml', '02-brunner-belegnachreichung.eml', '03-steiner-saldoanfrage.eml', '04-lindner-fehlender-beleg.eml', '05-hofer-mehrdeutig.eml']
  skills:     ['dummy-mapping', 'rechnungsrueckfrage']
  kontext:    ['belege', 'dummy-belege', 'kundenstamm.csv', 'palettenkonten', 'rechnungen', 'tourdaten']


In [11]:
# nest_asyncio optional (Jupyter kann top-level await), logging beruhigen
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

import logging
import structlog
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s | %(message)s")
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))


In [12]:
# Nur CLAUDE.md vorab laden — die Skills werden automatisch vom SkillManager
# unter `<project_root>/skills/<name>/SKILL.md` discovered (Fix #378, commit
# 0085b32). Der Factory hängt automatisch `activate_skill` an die Tools und
# injiziert eine AVAILABLE-SKILLS-Section in den System-Prompt.
CLAUDE_MD = (TUTTI_ROOT / "CLAUDE.md").read_text(encoding="utf-8")

# Welche Skills sind im Projekt verfügbar?
skill_dirs = sorted((TUTTI_ROOT / "skills").iterdir()) if (TUTTI_ROOT / "skills").is_dir() else []
print(f"CLAUDE.md          : {len(CLAUDE_MD):>6} chars")
print(f"Projekt-Skills     : {[d.name for d in skill_dirs if (d / 'SKILL.md').is_file()]}")

# Eine konkrete Test-Mail als Mission-Ziel
TARGET_MAIL = TUTTI_ROOT / "test-mails" / "01-mueller-bau-rechnungsfrage.eml"
assert TARGET_MAIL.is_file()

print(f"Test-Mail          : {TARGET_MAIL.name}")
print("\n--- Mail-Inhalt ---")
print(TARGET_MAIL.read_text(encoding="utf-8"))


CLAUDE.md          :   5374 chars
Projekt-Skills     : ['dummy-mapping', 'rechnungsrueckfrage']
Test-Mail          : 01-mueller-bau-rechnungsfrage.eml

--- Mail-Inhalt ---
From: Karin Berger <buchhaltung@mueller-bau.de>
To: palettenklaerung@btk-transport.example
Subject: Rueckfrage zu Rechnung RG-2026-04-0312
Date: Thu, 14 May 2026 09:12:00 +0200
Message-ID: <test-mail-01@mueller-bau.example>

Sehr geehrte Damen und Herren,

ich habe heute die Rechnung RG-2026-04-0312 ueber 266,56 EUR
erhalten. Wir kommen bei unserer internen Pruefung nicht ganz
auf diese Summe.

Koennten Sie uns die abgerechneten Bewegungen nochmal kurz
aufschluesseln? Konkret: Wir haben fuer April einen
Saldoendstand von 5 Paletten in unseren Unterlagen, das passt
auch. Aber die Tagesmiete erscheint uns hoch.

Vielen Dank im Voraus.

Mit freundlichen Gruessen
Karin Berger
Buchhaltung Mueller Bau GmbH



## 2. Run-Helper + RunRecord

Wir nutzen die bewährte `run()`-Funktion aus dem Playground und sammeln
parallel alle Events in einer `RunRecord`. Bewusst KEIN `context.snapshot()`
mitten im Streaming-Loop — das wechselt den asyncio-Context und sprengt den
Token-Ledger (`Token was created in a different Context`). Context inspizieren
wir **nach** dem Lauf direkt über `agent.context.snapshot(...)`.

Aus den gesammelten Events lassen sich ableiten:
- **Tool-Häufigkeiten** (per `Counter`)
- **Plan-Historie** (alle erfolgreichen `planner`-Tool-Resultate)
- **Final-Answer**
- **Lauf-Dauer** + Step-Count
- **Tool-Output-Größen** (Approximation der Context-Last)


In [13]:
import json
import time
from collections import Counter
from dataclasses import dataclass, field

from taskforce.core.domain.enums import EventType

INTERESTING = {
    EventType.STEP_START.value,
    EventType.TOOL_CALL.value,
    EventType.TOOL_RESULT.value,
    EventType.FINAL_ANSWER.value,
    EventType.COMPLETE.value,
    EventType.ERROR.value,
}


def full_event_message(ev) -> str:
    """Komplette, lesbare Message fuer ein ProgressUpdate."""
    details = ev.details or {}
    if ev.event_type_value == EventType.TOOL_CALL.value:
        args = details.get("args")
        if args:
            return (
                f"Calling: {details.get('tool', 'unknown')}\n"
                f"args: {json.dumps(args, ensure_ascii=False, indent=2, default=str)}"
            )
        return ev.message or ""
    if ev.event_type_value == EventType.TOOL_RESULT.value:
        status = "OK" if details.get("success") else "FAIL"
        output = details.get("output", "")
        if not isinstance(output, str):
            output = json.dumps(output, ensure_ascii=False, indent=2, default=str)
        return f"{status} {details.get('tool', 'unknown')}:\n{output}"
    return (ev.message or "").strip()


@dataclass
class RunRecord:
    """Was im Mission-Lauf passiert ist - Auswertung NACH run()."""
    events: list[dict] = field(default_factory=list)
    tool_calls: Counter = field(default_factory=Counter)
    plan_history: list[str] = field(default_factory=list)
    final_answer: str = ""
    duration: float = 0.0
    # Approximation: 1 Step = 1 "ReAct-Runde" = ein neuer tool_call-Burst
    # nach einem tool_result. step_start wird vom LeanAgent aktuell nicht
    # emittiert, deshalb leiten wir die Step-Zahl aus den Uebergaengen ab.
    step_count: int = 0
    llm_calls: int = 0  # = Anzahl assistant-Messages nach dem Lauf
    files_before: dict = field(default_factory=dict)
    files_after: dict = field(default_factory=dict)
    initial_system_prompt_chars: int = 0


def _snapshot_outputs(root, subdirs=("fall-log", "drafts")) -> dict:
    """Pfad -> (size, mtime) fuer alle Files in den Output-Verzeichnissen."""
    snap = {}
    for sub in subdirs:
        d = root / sub
        if not d.is_dir():
            continue
        for f in d.rglob("*"):
            if f.is_file():
                st = f.stat()
                snap[str(f.relative_to(root))] = (st.st_size, st.st_mtime)
    return snap


async def run(
    executor, agent, mission, *, project_root=None,
    initial_system_prompt_chars: int = 0, max_print_events: int = 40,
) -> RunRecord:
    """Mission ausfuehren, Events live mitloggen und in RunRecord sammeln.

    - KEIN break aus dem async for (Jupyter-Generator-Cleanup-Bug).
    - KEIN context.snapshot() waehrend des Loops (kann den asyncio-Context
      wechseln -> Token-Reset bricht).
    - Pre/Post-Datei-Snapshot, wenn project_root uebergeben wird.
    - Initial-System-Prompt-Groesse wird im Record festgehalten, damit
      5.4 die Skill-Injection sichtbar machen kann.
    """
    rec = RunRecord(initial_system_prompt_chars=initial_system_prompt_chars)
    if project_root is not None:
        rec.files_before = _snapshot_outputs(project_root)

    started = time.time()
    shown = 0
    truncated = False
    last_event_type = None

    async for ev in executor.execute_mission_streaming(mission=mission, agent=agent):
        et = ev.event_type_value
        details = ev.details or {}
        t = time.time() - started

        if et in {"step_start", "tool_call", "tool_result", "final_answer", "error"}:
            rec.events.append({
                "t": t,
                "event": et,
                "tool": details.get("tool"),
                "tool_args": details.get("args"),
                "tool_success": details.get("success"),
                "output": details.get("output") if et == "tool_result" else None,
                "message": ev.message,
            })

        # Step-Approximation: Uebergang (tool_result|start) -> tool_call = neuer Step
        if et == "tool_call" and last_event_type != "tool_call":
            rec.step_count += 1

        if et == "tool_call":
            rec.tool_calls[details.get("tool", "?")] += 1
        elif et == "tool_result":
            if details.get("tool") == "planner" and details.get("success"):
                out = details.get("output", "")
                if isinstance(out, str):
                    rec.plan_history.append(out[:3000])
        elif et == "final_answer":
            rec.final_answer = str(details.get("content") or ev.message or "")

        if et in INTERESTING:
            if shown < max_print_events:
                print(f"[{ev.event_type_value:14s}] {full_event_message(ev)}")
                shown += 1
            elif not truncated:
                print("... (Ausgabe gekappt; Mission laeuft im Hintergrund weiter)")
                truncated = True

        last_event_type = et

    rec.duration = time.time() - started

    try:
        rec.llm_calls = sum(
            1 for m in agent.context.messages if m.get("role") == "assistant"
        )
    except Exception:
        rec.llm_calls = 0

    if project_root is not None:
        rec.files_after = _snapshot_outputs(project_root)

    return rec


def file_diff_report(rec: RunRecord) -> dict:
    """Vergleicht files_before/files_after im RunRecord."""
    before, after = rec.files_before, rec.files_after
    new = sorted(set(after) - set(before))
    deleted = sorted(set(before) - set(after))
    common = set(before) & set(after)
    modified = sorted(p for p in common if before[p] != after[p])
    unchanged = sorted(p for p in common if before[p] == after[p])
    return {"new": new, "modified": modified, "deleted": deleted, "unchanged": unchanged}


print("run() + RunRecord + file_diff_report bereit.")


run() + RunRecord + file_diff_report bereit.


## 3. TuttiPaletti-Agent bauen


In [14]:
from taskforce.application.factory import AgentFactory
from taskforce.application.executor import AgentExecutor

factory = AgentFactory()
executor = AgentExecutor(factory=factory)


In [18]:
# TUTTI_ROOT als POSIX-String (forward slashes - auf Windows ok, ohne Escapes).
TUTTI_ROOT_STR = TUTTI_ROOT.as_posix()

# System-Prompt: nur CLAUDE.md + Pfad-/Tool-Hinweise.
TUTTI_SYSTEM_PROMPT = (
    "Du bist ein Sachbearbeiter fuer Palettenkonten. Deine Master-Anweisung "
    "(CLAUDE.md) ist unten direkt mitgegeben.\n\n"

    f"Projekt-Root: '{TUTTI_ROOT_STR}'. Nutze fuer file_read/file_write/grep "
    "IMMER ABSOLUTE Pfade unter diesem Root, z.B.\n"
    f"  '{TUTTI_ROOT_STR}/kontext/palettenkonten/K10847_mueller-bau.csv'\n"
    f"  '{TUTTI_ROOT_STR}/test-mails/01-mueller-bau-rechnungsfrage.eml'\n\n"

    "WICHTIG zur Effizienz:\n"
    "- LIES JEDE DATEI NUR EINMAL. Wenn du eine Datei bereits gelesen hast, "
    "der Inhalt steht in deinem Context.\n"
    "- Die Projekt-Struktur ist in CLAUDE.md beschrieben - du musst nicht "
    "suchen, sondern kannst direkt die Pfade konstruieren.\n"
    "- Nutze grep nur fuer Inhaltsuche, nicht fuer Datei-Discovery.\n\n"

    "Skills sind ueber das activate_skill-Tool verfuegbar.\n\n"

    "KEIN echtes Gmail: Drafts als Markdown nach "
    f"'{TUTTI_ROOT_STR}/drafts/<datum>-<kunde>.md'.\n\n"

    "=== CLAUDE.md ===\n"
    + CLAUDE_MD
)

# KEIN glob (failed bei absoluten Pfaden + redundant).
TUTTI_TOOLS = ["file_read", "file_write", "grep"]

tutti_agent = await factory.create_agent(
    system_prompt=TUTTI_SYSTEM_PROMPT,
    tools=TUTTI_TOOLS,
    persistence={"type": "file", "work_dir": str(TUTTI_ROOT / ".taskforce")},
    work_dir=str(TUTTI_ROOT),
    planning_strategy="plan_and_react",
    planning_strategy_params={"max_plan_steps": 8},
    max_steps=25,
)

# === Anti-Compression-Sturm Tuning (PATCH INTERNAL COMPONENTS) ===
# Defaults: summary_threshold=20, tool_result_store_threshold=2000.
#
# WICHTIG: Diese Werte werden in Agent.__init__ als private Attribute an
# MessageHistoryManager._summary_threshold und
# ToolExecutor._result_store_threshold weitergereicht. Setting auf der
# Top-Level-Agent-Instanz allein (tutti_agent.summary_threshold=...) hat
# KEINEN Effekt. Wir muessen die internen Komponenten direkt patchen.
tutti_agent.message_history_manager._summary_threshold = 80
tutti_agent.tool_executor._result_store_threshold = 6000
# Top-Level-Mirrors (nur fuer Print-Konsistenz):
tutti_agent.summary_threshold = 80
tutti_agent._tool_result_store_threshold = 6000

tool_names = [t for t in tutti_agent.tools]
has_skills = "activate_skill" in tool_names

print(f"tutti_agent OK")
print(f"  Tools                       : {tool_names}")
print(f"  activate_skill              : {'JA' if has_skills else 'NEIN'}")
print(f"  System-Prompt               : {len(TUTTI_SYSTEM_PROMPT)} chars")
print(f"  Strategy                    : {tutti_agent.planning_strategy.name}")
print(f"  Max steps                   : {tutti_agent.max_steps}")
print(f"  msg_hist._summary_threshold : {tutti_agent.message_history_manager._summary_threshold} (Default 20)")
print(f"  tool_exec._result_store_thr : {tutti_agent.tool_executor._result_store_threshold} (Default 2000)")


tutti_agent OK
  Tools                       : ['file_read', 'file_write', 'grep', 'activate_skill', 'planner']
  activate_skill              : JA
  System-Prompt               : 6310 chars
  Strategy                    : plan_and_react
  Max steps                   : 25
  msg_hist._summary_threshold : 80 (Default 20)
  tool_exec._result_store_thr : 6000 (Default 2000)


## 4. Mission ausführen + tracken


In [19]:
# OPTIONAL: Output-Verzeichnisse vor dem Lauf leeren, damit alte Files
# den Agenten nicht beeinflussen und der Datei-Diff nach dem Lauf
# eindeutig zeigt was *dieser* Lauf produziert hat.
# Auskommentieren wenn du Vorarbeit absichtlich erhalten willst.
import shutil

RESET_BEFORE_RUN = True   # auf False setzen, um Outputs zu behalten

if RESET_BEFORE_RUN:
    for sub in ("fall-log", "drafts"):
        d = TUTTI_ROOT / sub
        if d.exists():
            shutil.rmtree(d)
        d.mkdir()
    print(f"Reset done: {[sub for sub in ('fall-log','drafts')]}")
else:
    print("Skipped reset - existierende Outputs bleiben (Vorsicht: koennen den Agent beeinflussen)")


Reset done: ['fall-log', 'drafts']


In [20]:
mail_abs = TARGET_MAIL.as_posix()
mission = (
    f"Bearbeite die Mail '{mail_abs}' gemaess Workflow 1 (Mail-Loop, "
    "Rechnungsrueckfrage). Schreibe die Falldatei nach "
    f"'{TUTTI_ROOT_STR}/fall-log/' und einen Antwort-Draft als Markdown nach "
    f"'{TUTTI_ROOT_STR}/drafts/'. Am Ende eine kurze Zusammenfassung."
)

record = await run(
    executor, tutti_agent, mission,
    project_root=TUTTI_ROOT,                              # Pre/Post-Datei-Snapshot
    initial_system_prompt_chars=len(TUTTI_SYSTEM_PROMPT), # fuer Skill-Injection-Diagnose
    max_print_events=40,
)

print(f"\n{'='*60}")
print(
    f"Duration: {record.duration:.1f}s | "
    f"Steps (Bursts): {record.step_count} | "
    f"LLM-Calls: {record.llm_calls} | "
    f"Tool calls: {sum(record.tool_calls.values())}"
)
print(f"\nFinal answer (erste 800 chars):\n")
print(record.final_answer[:800])


[tool_call     ] Calling: activate_skill
args: {
  "skill_name": "rechnungsrueckfrage"
}
[tool_result   ] OK activate_skill:
{"success": true, "skill_name": "rechnungsrueckfrage", "has_workflow": false, "message": "Skill 'rechnungsrueckfrage' activated. Follow the skill instructions in the system prompt."}
[tool_call     ] Calling: file_read
args: {
  "path": "C:/Users/rudi/source/repos/TuttiPaletti/test-mails/01-mueller-bau-rechnungsfrage.eml",
  "encoding": "utf-8",
  "max_size_mb": 2
}
[tool_result   ] OK file_read:
{"success": true, "content": "From: Karin Berger <buchhaltung@mueller-bau.de>\nTo: palettenklaerung@btk-transport.example\nSubject: Rueckfrage zu Rechnung RG-2026-04-0312\nDate: Thu, 14 May 2026 09:12:00 +0200\nMessage-ID: <test-mail-01@mueller-bau.example>\n\nSehr geehrte Damen und Herren,\n\nich habe heute die Rechnung RG-2026-04-0312 ueber 266,56 EUR\nerhalten. Wir kommen bei unserer internen Pruefung nicht ganz\nauf diese Summe.\n\nKoennten Sie uns die abgerechneten 

In [21]:
# Datei-Diff: was hat *dieser* Lauf an Output-Files produziert?
# Deckt die Halluzinations-Variante "Agent claimt Datei geschrieben,
# aber Datei stammt aus frueherem Lauf" auf.
diff = file_diff_report(record)

print("=== Datei-Diff (fall-log/ + drafts/) ===")
print(f"  NEU geschrieben   : {len(diff['new']):>2}")
for p in diff["new"]:
    size = record.files_after[p][0]
    print(f"     + {p}  ({size} bytes)")

print(f"  MODIFIZIERT       : {len(diff['modified']):>2}")
for p in diff["modified"]:
    old_size = record.files_before[p][0]
    new_size = record.files_after[p][0]
    print(f"     M {p}  ({old_size} -> {new_size} bytes)")

print(f"  UNVERAENDERT      : {len(diff['unchanged']):>2}")
for p in diff["unchanged"]:
    print(f"     . {p}")

if diff["deleted"]:
    print(f"  GELOESCHT         : {len(diff['deleted']):>2}")
    for p in diff["deleted"]:
        print(f"     - {p}")

# Sanity check
file_write_count = record.tool_calls.get("file_write", 0)
edit_count = record.tool_calls.get("edit", 0)
total_writes = file_write_count + edit_count
total_changed = len(diff["new"]) + len(diff["modified"])

print(f"\n=== Sanity-Check ===")
print(f"  file_write Calls         : {file_write_count}")
print(f"  edit Calls               : {edit_count}")
print(f"  Neue/modifizierte Files  : {total_changed}")
if total_writes == 0 and total_changed > 0:
    print("  WARNUNG: Files geaendert, aber keine write/edit Tool-Calls -")
    print("  vermutlich vom Tool-Result-Store oder einem frueheren Lauf.")
elif total_writes > 0 and total_changed == 0:
    print("  WARNUNG: write/edit Calls, aber keine Datei-Aenderungen -")
    print("  HALLUZINATION oder Workspace-Pfad-Problem moeglich.")
elif total_changed == 0:
    print("  WARNUNG: Keine neuen/modifizierten Output-Files -")
    print("  pruefe Final-Answer auf Halluzination.")
else:
    print("  OK")


=== Datei-Diff (fall-log/ + drafts/) ===
  NEU geschrieben   :  2
     + drafts\2026-05-17-mueller-bau-gmbh.md  (1133 bytes)
     + fall-log\2026-05-17-mueller-bau-gmbh.md  (2519 bytes)
  MODIFIZIERT       :  0
  UNVERAENDERT      :  0

=== Sanity-Check ===
  file_write Calls         : 2
  edit Calls               : 0
  Neue/modifizierte Files  : 2
  OK


## 5. Reports

Drei Ansichten auf den Run:

- **5.1** Tool-Häufigkeit + Plan-Snapshots + generierte Artefakte
- **5.2** Context-Evolution (Messages und Chars pro Step)
- **5.3** Tool-Result-Previews — was kam zurück, wovon kann das LLM beim nächsten Step profitieren?
- **5.4** Aktueller Context (letzte N Messages) zum Debugging


In [22]:
# 5.1 Tool-Aufrufe + Plan-Trace + generierte Artefakte
print("Tool-Aufrufe (Häufigkeit):")
for tool, n in record.tool_calls.most_common():
    print(f"  {tool:14s} {n:>3}x")

print(f"\nTotal Events:                       {len(record.events)}")
print(f"Plan-Snapshots (planner-Tool):      {len(record.plan_history)}")

if record.plan_history:
    print("\n--- Letzter Plan-Snapshot ---")
    print(record.plan_history[-1])
else:
    print("\n(keine Planner-Tool-Aufrufe — strategieabhängig.)")

print("\n--- Generated Artifacts in TuttiPaletti ---")
for sub in ("fall-log", "drafts"):
    d = TUTTI_ROOT / sub
    if d.is_dir():
        for f in sorted(d.iterdir()):
            if f.is_file():
                print(f"  {sub}/{f.name}  ({f.stat().st_size} bytes)")


Tool-Aufrufe (Häufigkeit):
  file_read       12x
  planner          6x
  grep             3x
  file_write       2x
  activate_skill   1x

Total Events:                       49
Plan-Snapshots (planner-Tool):      6

--- Letzter Plan-Snapshot ---
[x] 1. Read the email file and identify sender, subject, customer, invoice reference, and the exact request.
[x] 2. Search the relevant context files under C:/Users/rudi/source/repos/TuttiPaletti/kontext/ for the customer’s pallet account, related movements, and any matching delivery documents.
[x] 3. Derive the balance explanation from the evidence and determine whether the case is clear or needs escalation (missing evidence / low confidence / ambiguity).
[x] 4. Write the case log markdown file to C:/Users/rudi/source/repos/TuttiPaletti/fall-log/ with original text, classification, evidence, balance derivation, and open questions.
[x] 5. Create the reply draft markdown file in C:/Users/rudi/source/repos/TuttiPaletti/drafts/ with a polite answe

In [23]:
# 5.2 Event-Timeline + approximative Context-Last (kumulierte Tool-Output-Größen)
#
# Ohne mid-flight context.snapshot() haben wir keine exakte Char-Anzahl pro
# Step — als Proxy summieren wir die Größen der bisher gesehenen Tool-Outputs.
# Das ist eine Untergrenze; die echte Context-Größe inkl. System-Prompt +
# Assistant-Tokens ist höher, aber das Wachstumsmuster ist identisch.
print(f"{'t [s]':>6} {'step':>5} {'event':14s} {'tool':10s} {'note':60s}")
print("-" * 100)
cum_chars = 0
step = 0
for e in record.events:
    et = e["event"]
    if et == "step_start":
        step += 1
        note = "(neue ReAct-Runde)"
    elif et == "tool_call":
        args_str = json.dumps(e.get("tool_args") or {}, ensure_ascii=False, default=str)
        note = args_str[:60]
    elif et == "tool_result":
        out = e.get("output") or ""
        if not isinstance(out, str):
            out = json.dumps(out, default=str)
        cum_chars += len(out)
        status = "OK" if e["tool_success"] else "FAIL"
        note = f"{status} | out {len(out):>5} chars | sum {cum_chars:>6}"
    else:
        note = (e.get("message") or "")[:60]
    print(f"{e['t']:6.2f} {step:>5} {et:14s} {(e.get('tool') or ''):10s} {note}")


 t [s]  step event          tool       note                                                        
----------------------------------------------------------------------------------------------------
  5.08     0 tool_call      activate_skill {"skill_name": "rechnungsrueckfrage"}
  5.08     0 tool_result    activate_skill OK | out   182 chars | sum    182
  6.31     0 tool_call      file_read  {"path": "C:/Users/rudi/source/repos/TuttiPaletti/test-mails
  6.32     0 tool_result    file_read  OK | out   880 chars | sum   1062
  8.48     0 tool_call      grep       {"pattern": "mueller-bau|Mueller Bau|Müller Bau", "path": "C
  8.48     0 tool_call      grep       {"pattern": "mueller-bau|Mueller Bau|Müller Bau", "path": "C
  8.50     0 tool_result    grep       OK | out  1068 chars | sum   2130
  8.50     0 tool_result    grep       OK | out   453 chars | sum   2583
 10.40     0 tool_call      grep       {"pattern": "K10847", "path": "C:/Users/rudi/source/repos/Tu
 10.40     0 tool_call

In [24]:
# 5.3 Tool-Call -> Tool-Result paarweise
# Annotation: file_read auf "tool_results/results/<uuid>.json" = ADR-025
# Tool-Result-Store-Lookup (grosse fruehere Tool-Outputs liegen auf Disk).
for e in record.events:
    if e["event"] == "tool_call":
        args = e.get("tool_args") or {}
        args_str = json.dumps(args, ensure_ascii=False, default=str)
        annot = ""
        if e["tool"] == "file_read" and "tool_results/results/" in str(args.get("path", "")):
            annot = "  [ADR-025 Tool-Result-Store-Lookup]"
        print(f"\n  -> {e['tool']}({args_str[:150]}){annot}")
    elif e["event"] == "tool_result":
        status = "OK" if e["tool_success"] else "FAIL"
        out = e.get("output") or ""
        if not isinstance(out, str):
            out = json.dumps(out, default=str)
        preview = out[:200].replace("\n", " ")
        print(f"     {status}: {preview}")



  -> activate_skill({"skill_name": "rechnungsrueckfrage"})
     OK: {"success": true, "skill_name": "rechnungsrueckfrage", "has_workflow": false, "message": "Skill 'rechnungsrueckfrage' activated. Follow the skill instructions in the system prompt."}

  -> file_read({"path": "C:/Users/rudi/source/repos/TuttiPaletti/test-mails/01-mueller-bau-rechnungsfrage.eml", "encoding": "utf-8", "max_size_mb": 2})
     OK: {"success": true, "content": "From: Karin Berger <buchhaltung@mueller-bau.de>\nTo: palettenklaerung@btk-transport.example\nSubject: Rueckfrage zu Rechnung RG-2026-04-0312\nDate: Thu, 14 May 2026 09:12

  -> grep({"pattern": "mueller-bau|Mueller Bau|Müller Bau", "path": "C:/Users/rudi/source/repos/TuttiPaletti/kontext", "glob": "**/*", "output_mode": "files_wit)

  -> grep({"pattern": "mueller-bau|Mueller Bau|Müller Bau", "path": "C:/Users/rudi/source/repos/TuttiPaletti/kontext/kundenstamm.csv", "output_mode": "content",)
     OK: {"success": true, "files": ["C:\\Users\\rudi\\sour

In [25]:
# 5.4 Context NACH dem Lauf
#
# Zwei Views + Skill-Injection-Diagnose:
#   - snap        : strukturierter Snapshot mit Token-Counts pro Section
#   - raw_messages: agent.context.messages - die echte Message-Liste
#   - Skill-Diff  : initialer System-Prompt vs jetzige Groesse zeigt, ob/wie
#                   viel ein Skill in den Prompt injiziert wurde
from collections import defaultdict

snap = tutti_agent.context.snapshot(include_content=True)
raw_messages = tutti_agent.context.messages

print(f"=== Snapshot (Token-View) ===")
print(f"  total_tokens        : {snap.total_tokens:>6}")
print(f"  max_tokens          : {snap.max_tokens:>6}")
print(f"  utilization         : {snap.utilization_percent:>6.1f}%")
print(f"  system_prompt items : {len(snap.system_prompt):>3}  ({sum(i.tokens for i in snap.system_prompt):>5} tokens)")
print(f"  messages items      : {len(snap.messages):>3}  ({sum(i.tokens for i in snap.messages):>5} tokens)")
print(f"  memory items        : {len(snap.memory):>3}  ({sum(i.tokens for i in snap.memory):>5} tokens)")
print(f"  skills items        : {len(snap.skills):>3}  ({sum(i.tokens for i in snap.skills):>5} tokens)")
print(f"  tools items         : {len(snap.tools):>3}  ({sum(i.tokens for i in snap.tools):>5} tokens)")
print(f"  sub_agents          : {len(snap.sub_agents):>3}")
print("  Note: Bei has_workflow=False Skills wird der Skill-Inhalt in die")
print("        system_prompt-Section gemerged, NICHT in 'skills items'.")

print(f"\n=== Raw Messages (was geht ans LLM) ===")
total_chars = sum(len(str(m.get("content", ""))) for m in raw_messages)
print(f"  {len(raw_messages)} messages, {total_chars} chars total\n")

by_role = defaultdict(lambda: [0, 0])
for m in raw_messages:
    by_role[m.get("role", "?")][0] += 1
    by_role[m.get("role", "?")][1] += len(str(m.get("content", "")))
print("  Per-Role-Verteilung:")
for role, (n, c) in sorted(by_role.items()):
    print(f"    {role:9s} {n:>3} msgs  {c:>7} chars")

# Skill-Injection-Diagnose
print(f"\n=== Skill-Injection-Diagnose ===")
system_now = by_role.get("system", [0, 0])[1]
initial = record.initial_system_prompt_chars
delta = system_now - initial
print(f"  System-Prompt initial      : {initial:>6} chars (TUTTI_SYSTEM_PROMPT)")
print(f"  System-Messages jetzt total: {system_now:>6} chars (ueber {by_role.get('system',[0])[0]} msgs)")
print(f"  Delta                      : {delta:+>6} chars")

skill_calls = [
    e for e in record.events
    if e["event"] == "tool_call" and e["tool"] == "activate_skill"
]
if skill_calls:
    names = [str((e.get("tool_args") or {}).get("skill_name", "?")) for e in skill_calls]
    print(f"  activate_skill Calls       : {len(skill_calls)}x  ({names})")
    if delta > 1000:
        print(f"  -> Skill wurde injiziert (~{delta} chars dazu)")
    elif delta > 0:
        print(f"  -> Skill aktiviert, aber Injektion klein - moeglich nur Hinweis")
    else:
        print(f"  -> Hm, keine Vergroesserung trotz activate_skill - bug?")
else:
    print(f"  activate_skill Calls       : 0x")
    if delta > 1000:
        print(f"  -> Trotzdem Wachstum (+{delta}) - vermutlich Conversation/Memory")

# Letzte 8 Messages als Vorschau
print("\n  --- Letzte 8 Messages ---")
for m in raw_messages[-8:]:
    content = str(m.get("content", "")).replace("\n", " ")
    role = m.get("role", "?")
    print(f"  [{role:9s}] {len(content):>5} chars: {content[:180]}...")


=== Snapshot (Token-View) ===
  total_tokens        :  10926
  max_tokens          : 100000
  utilization         :   10.9%
  system_prompt items :   1  ( 3581 tokens)
  messages items      :  36  ( 6142 tokens)
  memory items        :   0  (    0 tokens)
  skills items        :   0  (    0 tokens)
  tools items         :   5  ( 1203 tokens)
  sub_agents          :   0
  Note: Bei has_workflow=False Skills wird der Skill-Inhalt in die
        system_prompt-Section gemerged, NICHT in 'skills items'.

=== Raw Messages (was geht ans LLM) ===
  37 messages, 29804 chars total

  Per-Role-Verteilung:
    assistant  10 msgs      800 chars
    system      1 msgs    14325 chars
    tool       24 msgs    14092 chars
    user        2 msgs      587 chars

=== Skill-Injection-Diagnose ===
  System-Prompt initial      :   6310 chars (TUTTI_SYSTEM_PROMPT)
  System-Messages jetzt total:  14325 chars (ueber 1 msgs)
  Delta                      : ++8015 chars
  activate_skill Calls       : 1x  (['rechn

## 6. Strategien vergleichen (optional)

Selbe Mission, andere Planning-Strategie. Zweiter vollständiger Lauf — dauert
entsprechend. Wir bauen den Agent jeweils frisch (eigenes `work_dir` damit
die State-Files nicht kollidieren) und vergleichen am Ende Kennzahlen.


In [27]:
async def run_with_strategy(strategy: str, params: dict | None = None) -> RunRecord:
    tutti_agent = await factory.create_agent(
        system_prompt=TUTTI_SYSTEM_PROMPT,
        tools=TUTTI_TOOLS,
        persistence={"type": "file", "work_dir": str(TUTTI_ROOT / f".taskforce_{strategy}")},
        work_dir=str(TUTTI_ROOT),
        planning_strategy=strategy,
        planning_strategy_params=params,
        max_steps=25,
    )

    # === Anti-Compression-Sturm Tuning (PATCH INTERNAL COMPONENTS) ===
    # Defaults: summary_threshold=20, tool_result_store_threshold=2000.
    #
    # WICHTIG: Diese Werte werden in Agent.__init__ als private Attribute an
    # MessageHistoryManager._summary_threshold und
    # ToolExecutor._result_store_threshold weitergereicht. Setting auf der
    # Top-Level-Agent-Instanz allein (tutti_agent.summary_threshold=...) hat
    # KEINEN Effekt. Wir muessen die internen Komponenten direkt patchen.
    tutti_agent.message_history_manager._summary_threshold = 80
    tutti_agent.tool_executor._result_store_threshold = 6000
    # Top-Level-Mirrors (nur fuer Print-Konsistenz):
    tutti_agent.summary_threshold = 80
    tutti_agent._tool_result_store_threshold = 6000
    
    tool_names = [t for t in tutti_agent.tools]
    has_skills = "activate_skill" in tool_names
    
    print(f"tutti_agent OK")
    print(f"  Tools                       : {tool_names}")
    print(f"  activate_skill              : {'JA' if has_skills else 'NEIN'}")
    print(f"  System-Prompt               : {len(TUTTI_SYSTEM_PROMPT)} chars")
    print(f"  Strategy                    : {tutti_agent.planning_strategy.name}")
    print(f"  Max steps                   : {tutti_agent.max_steps}")
    print(f"  msg_hist._summary_threshold : {tutti_agent.message_history_manager._summary_threshold} (Default 20)")
    print(f"  tool_exec._result_store_thr : {tutti_agent.tool_executor._result_store_threshold} (Default 2000)")
    
    print(f"\n>>> Strategy: {strategy}")
    rec = await run(
        executor, tutti_agent, mission,
        project_root=TUTTI_ROOT,
        initial_system_prompt_chars=len(TUTTI_SYSTEM_PROMPT),
        max_print_events=12,
    )
    await tutti_agent.close()
    return rec

# record (oben) = plan_and_react. Wir vergleichen mit native_react.
rec_react = await run_with_strategy("native_react")

print("\n=== Vergleich ===")
print(f"  {'strategy':18s}  {'bursts':>6} {'llm':>4} {'tools':>6} {'plans':>5}  {'duration':>9}")
print(f"  {'-'*16:18s}  {'-'*6:>6} {'-'*4:>4} {'-'*6:>6} {'-'*5:>5}  {'-'*9:>9}")
for label, r in [("plan_and_react", record), ("native_react", rec_react)]:
    print(
        f"  {label:18s}  {r.step_count:>6} {r.llm_calls:>4} "
        f"{sum(r.tool_calls.values()):>6} {len(r.plan_history):>5}  "
        f"{r.duration:>8.2f}s"
    )

# File-Diff-Vergleich
print("\n=== File-Diff-Vergleich ===")
for label, r in [("plan_and_react", record), ("native_react", rec_react)]:
    d = file_diff_report(r)
    print(f"  {label:18s}  neu={len(d['new'])}  mod={len(d['modified'])}  unchanged={len(d['unchanged'])}")


tutti_agent OK
  Tools                       : ['file_read', 'file_write', 'grep', 'activate_skill', 'planner']
  activate_skill              : JA
  System-Prompt               : 6310 chars
  Strategy                    : native_react
  Max steps                   : 25
  msg_hist._summary_threshold : 80 (Default 20)
  tool_exec._result_store_thr : 6000 (Default 2000)

>>> Strategy: native_react
[tool_call     ] Calling: activate_skill
args: {
  "skill_name": "rechnungsrueckfrage"
}
[tool_result   ] OK activate_skill:
{"success": true, "skill_name": "rechnungsrueckfrage", "has_workflow": false, "message": "Skill 'rechnungsrueckfrage' activated. Follow the skill instructions in the system prompt."}
[tool_call     ] Calling: file_read
args: {
  "path": "C:/Users/rudi/source/repos/TuttiPaletti/test-mails/01-mueller-bau-rechnungsfrage.eml",
  "encoding": "utf-8",
  "max_size_mb": 2
}
[tool_result   ] OK file_read:
{"success": true, "content": "From: Karin Berger <buchhaltung@mueller-bau.de>

## 7. Aufräumen


In [ ]:
await tutti_agent.close()
print("tutti_agent geschlossen.")


## Ideen für weitere Experimente

- **Anderes Tool-Set**: ohne `python`, oder mit `edit` zusätzlich für gezielte
  Korrekturen am Draft
- **Anderer System-Prompt**: nur Bootstrap ("Lies CLAUDE.md") statt eingebettetem
  Inhalt — testet, ob der Agent das Skill-System wirklich selbständig nutzt
- **Mehrere Mails am Stück**: `for mail in (TUTTI_ROOT / 'test-mails').iterdir(): ...`
  und für jeden Lauf einen neuen Tracker behalten → Vergleichstabelle
- **Andere Strategie**: `spar` mit `reflect_every_step=True` für mehr
  Selbstkritik (langsamer, aber oft korrekter)
- **Engere ContextPolicy**: testet, wie der Agent mit weniger Context-Budget umgeht
- **Sub-Agents**: Workflow 2 (Dummy-Mapping) erlaubt Parallel-Sub-Agents — das
  ist eine andere Klasse von Analyse (Parallelität, Synthese der Ergebnisse)
